# OpenTorpedo — Teknofest optimizasyon

Bu notebook [opentorpedo](https://github.com/efeerdogmus0/opentorpedo) reposunu Colab'da çalıştırır.
Ağır simülasyon **senin PC'de değil**, Google sunucusunda koşar.

---

## Ne yapacaksın? (4 kod hücresi)

| Adım | Hücre | Süre |
|------|-------|------|
| 1 | Repoyu GitHub'dan indir | ~30 sn |
| 2 | Kütüphane kontrolü | ~5 sn |
| 3 | **Optimizasyon** (aşağıda `PRESET` seç) | 2–25 dk |
| 4 | Sonuç + JSON indir | ~5 sn |

Menü: **Çalışma zamanı → Tümünü çalıştır** (üstten alta).

### Preset (3. hücrede)

| `PRESET` | Deneme | Süre |
|----------|--------|------|
| `"quick"` | ~96 | ~2 dk |
| `"fast"` | ~160 | ~3 dk |
| `"medium"` | ~1900 | ~8 dk |
| `"full"` | ~6900 | ~15–25 dk |

İlk kez: `quick` veya `fast`. Gerçek arama: `full`.

### Sekmeyi kapatma

3. hücre çalışırken **tarayıcı sekmesini kapatma**, bilgisayarı uyutma. Kesilirse 1→4'ü yeniden çalıştır.

**Kısıtlar:** kütle ≤500 g · tel ≤2 mm · bobin ≤16 mm · yay boyu ≤60 mm

In [ ]:
# === ADIM 1/4: Repoyu klonla ===
import os
import sys
from pathlib import Path

WORKDIR = Path("/content/opentorpedo")
GIT_URL = "https://github.com/efeerdogmus0/opentorpedo.git"

if WORKDIR.exists():
    !rm -rf "{WORKDIR}"

!git clone --depth 1 "{GIT_URL}" "{WORKDIR}"
os.chdir(WORKDIR)
sys.path.insert(0, str(WORKDIR))

print("Klonlandı:", WORKDIR)
print("Dosyalar:", [p.name for p in WORKDIR.iterdir() if p.is_file()][:8], "…")

In [ ]:
# === ADIM 2/4: Bağımlılık kontrolü ===
import numpy as np
import scipy
print("numpy", np.__version__, "| scipy", scipy.__version__)
print("Hazır.")

In [ ]:
# === ADIM 3/4: Optimizasyon ===
from teknofest.grid_search import run_grid_search, count_combos

PRESET = "full"   # quick | fast | medium | full

print(f"Preset: {PRESET} | {count_combos(PRESET)} kombinasyon")
print("Başlıyor… (ilerleme satırları aşağıda)\n")

result = run_grid_search(
    PRESET,
    root=WORKDIR,
    progress_every=100,
)

In [ ]:
# === ADIM 4/4: Özet ve indirme ===
from IPython.display import display, JSON
from google.colab import files

display(JSON(result))

out = WORKDIR / "configs" / "teknofest_optimized.json"
files.download(str(out))

b, s, h = result["ballast"], result["spring"], result["hull"]
print("\n" + "=" * 50)
print("ÜRETİM ÖZETİ (simülasyon önerisi)")
print("=" * 50)
print(f"Max hız (sim)     : {result['max_speed_m_s']} m/s")
print(f"Toplam kütle      : {result['total_mass_g']} g  (≤500 g)")
print(f"Ağırlık merkezi   : {result['cog_cm']} cm (burundan)")
print(f"Gövde infill      : %{h['infill_percent']}  ({h['mass_g']} g)")
print(f"Balast            : {b['mass_g']} g @ {b['position_cm']} cm")
print(f"Yay tel çapı      : {s['wire_diameter_mm']} mm")
print(f"Yay bobin çapı    : {s['coil_diameter_mm']} mm")
print(f"Sarım sayısı      : {s['active_coils']}")
print(f"Serbest uzunluk   : {s['free_length_mm']} mm")
print(f"Sıkıştırma        : {s['compression_mm']} mm")
print(f"Kuvvet            : {s['force_N']} N")
print(f"Δv (yay)          : {s['delta_v_m_s']} m/s")
print("=" * 50)
print("JSON indirildi → PC'de configs/ klasörüne kaydet.")